# Industrial Visual Anomaly Detection — Eight-Model Research Benchmark

This notebook benchmarks eight **research implementations or simplified approximations** on MVTec AD. It is *not* an industrial-grade quality-control product. Read the repository's `README.md`, `docs/technical-review.md`, and `datasets/README.md` before running.

**Environment:** Python 3.11+ recommended, install dependencies using `pip install -r requirements.txt`. `torch` and `torchvision` must be compatible. Download MVTec AD separately, then set `MVTEC_ROOT` to the directory containing category folders. The dataset is non-commercially licensed and is **not included** in this repository.

**Important:** This public notebook starts with empty execution output and corrects some evaluation errors in the supplied artifact. Any previously reported figures should be treated as *historical, unverified outputs*, not results reproduced by this version.


## Dataset setup

- Official MVTec AD dataset: https://www.mvtec.com/research-teaching/datasets/mvtec-ad
- Kaggle mirror linked by original artifact: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
- Local structure: `$MVTEC_ROOT/bottle/train/good/...`, `$MVTEC_ROOT/bottle/test/...`, `$MVTEC_ROOT/bottle/ground_truth/...`

Example: `export MVTEC_ROOT=/path/to/mvtec_ad` (macOS/Linux) or `$env:MVTEC_ROOT='C:\path\to\mvtec_ad'` (PowerShell). You may also set `RESULTS_DIR`. Dataset downloading is intentionally kept outside the notebook.


In [ ]:
import os
import time
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)
from sklearn.manifold import TSNE
from scipy.ndimage import label as nd_label
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display


In [ ]:
# CONFIGURATION & REPRODUCIBILITY

CONFIG = {
    "seed": 42,
    "img_size": 256,
    "batch_size": 32,
    "epochs": 5,
    "lr": 1e-3,
    "num_workers": 1,
    "max_tsne_samples": 2000,
    "hybrid_alpha": 0.5,
    "use_amp": torch.cuda.is_available(),
    "dataset_path": os.environ.get("MVTEC_ROOT", "./datasets/mvtec_ad"),
    "results_dir": os.environ.get("RESULTS_DIR", "./results"),
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225]
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)
os.makedirs(CONFIG["results_dir"], exist_ok=True)

In [ ]:
# DATASET

class MVTecDataset(Dataset):
    """Unified MVTec AD Dataset optimized for low memory during training."""
    def __init__(self, root_dir, class_name, transform=None, mask_transform=None, mode="train"):
        self.root_dir = os.path.join(root_dir, class_name)
        self.class_name = class_name
        self.transform = transform
        self.mask_transform = mask_transform
        self.mode = mode
        self.samples = []

        sub_dir = os.path.join(self.root_dir, self.mode)
        gt_dir = os.path.join(self.root_dir, "ground_truth")

        if os.path.exists(sub_dir):
            for defect_type in os.listdir(sub_dir):
                defect_path = os.path.join(sub_dir, defect_type)
                if not os.path.isdir(defect_path):
                    continue

                for img_name in os.listdir(defect_path):
                    img_path = os.path.join(defect_path, img_name)
                    is_defect = 0 if defect_type == "good" else 1

                    mask_path = None
                    if is_defect and self.mode == "test":
                        base_name = os.path.splitext(img_name)[0]
                        mask_path = os.path.join(gt_dir, defect_type, f"{base_name}_mask.png")
                        if not os.path.exists(mask_path):
                            mask_path = os.path.join(gt_dir, defect_type, f"{base_name}.png")

                    self.samples.append((img_path, mask_path, is_defect, defect_type))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, is_defect, defect_type = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.mode == "train":
            return image

        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
            if self.mask_transform:
                mask = self.mask_transform(mask)
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros((1, image.shape[1], image.shape[2]), dtype=torch.float32)

        return {
            "image": image,
            "label": is_defect,
            "mask": mask,
            "path": img_path,
            "class_name": self.class_name,
            "defect_type": defect_type
        }

def get_mvtec_dataset(root_dir, classes, transform, mask_transform=None, mode="train"):
    if isinstance(classes, str):
        classes = [classes]
    datasets = [MVTecDataset(root_dir, c, transform, mask_transform, mode) for c in classes if os.path.exists(os.path.join(root_dir, c))]
    if not datasets:
        raise ValueError(f"No valid dataset classes found under {root_dir}")
    return ConcatDataset(datasets) if len(datasets) > 1 else datasets[0]

In [ ]:
# TRANSFORMS & DATALOADERS

train_transform = transforms.Compose([

    # Resize slightly larger before random crop
    transforms.Resize((286, 286)),

    # Random Crop to 256 × 256
    transforms.RandomCrop(CONFIG["img_size"]),

    # Random Horizontal Flip
    transforms.RandomHorizontalFlip(p=0.5),

    # Random Rotation
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR
    ),

    # Brightness Augmentation
    transforms.ColorJitter(
        brightness=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=CONFIG["imagenet_mean"],
        std=CONFIG["imagenet_std"]
    )

])

test_transform = transforms.Compose([

    transforms.Resize(
        (CONFIG["img_size"], CONFIG["img_size"])
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=CONFIG["imagenet_mean"],
        std=CONFIG["imagenet_std"]
    )

])

mask_transform = transforms.Compose([

    transforms.Resize(
        (CONFIG["img_size"], CONFIG["img_size"]),
        interpolation=transforms.InterpolationMode.NEAREST
    ),

    transforms.ToTensor()

])

if not os.path.isdir(CONFIG["dataset_path"]):
    raise FileNotFoundError(
        f"MVTec AD dataset missing at {CONFIG['dataset_path']!r}. "
        "Set MVTEC_ROOT to a folder containing the MVTec category directories."
    )

ALL_CLASSES = sorted([

    folder
    for folder in os.listdir(CONFIG["dataset_path"])
    if os.path.isdir(
        os.path.join(CONFIG["dataset_path"], folder)
    )

])

print("=" * 60)
print(f"Total Classes Loaded : {len(ALL_CLASSES)}")
print(ALL_CLASSES)
print("=" * 60)


# Create Combined Training Dataset


train_dataset = get_mvtec_dataset(

    root_dir=CONFIG["dataset_path"],
    classes=ALL_CLASSES,
    transform=train_transform,
    mode="train"

)


# Create Combined Testing Dataset


test_dataset = get_mvtec_dataset(

    root_dir=CONFIG["dataset_path"],
    classes=ALL_CLASSES,
    transform=test_transform,
    mask_transform=mask_transform,
    mode="test"

)


# Create DataLoaders


train_loader = DataLoader(

    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=CONFIG["num_workers"] > 0

)

test_loader = DataLoader(

    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=CONFIG["num_workers"] > 0

)


# Dataset Summary


print(f"Training Images : {len(train_dataset)}")
print(f"Testing Images  : {len(test_dataset)}")
print(f"Train Batches   : {len(train_loader)}")
print(f"Test Batches    : {len(test_loader)}")

calibration_dataset = get_mvtec_dataset(
    root_dir=CONFIG["dataset_path"],
    classes=ALL_CLASSES,
    transform=test_transform,
    mode="train",
)
calibration_loader = DataLoader(
    calibration_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=CONFIG["num_workers"] > 0,
)


In [ ]:
# BASE MODEL INTERFACE

class BaseAnomalyModel(nn.Module):
    """Standardized Abstract Base Interface for all thesis models."""
    def forward(self, x: torch.Tensor):
        raise NotImplementedError

    def compute_loss(self, batch) -> torch.Tensor:
        raise NotImplementedError

    def predict(self, x: torch.Tensor) -> dict:
        raise NotImplementedError

    def get_model_size_mb(self) -> float:
        param_size = sum(p.numel() * p.element_size() for p in self.parameters())
        buffer_size = sum(b.numel() * b.element_size() for b in self.buffers())
        return (param_size + buffer_size) / (1024 ** 2)

In [ ]:
# CONV AUTOENCODER MODEL

class ConvAutoencoder(BaseAnomalyModel):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(True)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1)
        )

    def forward(self, x: torch.Tensor):
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return recon, latent

    def compute_loss(self, batch) -> torch.Tensor:
        images = batch if isinstance(batch, torch.Tensor) else batch["image"]
        recon, _ = self.forward(images)
        return F.mse_loss(recon, images)

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            recon, latent = self.forward(x)
            pixel_err = torch.mean((x - recon) ** 2, dim=1, keepdim=True)
            image_err = pixel_err.view(pixel_err.size(0), -1).max(dim=1)[0]
            pooled_embed = self.pool(latent).squeeze(-1).squeeze(-1)

            return {
                "image_scores": image_err.cpu().numpy(),
                "pixel_scores": pixel_err.squeeze(1).cpu().numpy(),
                "reconstruction": recon.cpu().numpy(),
                "embeddings": pooled_embed.cpu().numpy()
            }

In [ ]:
# GAN ANOMALY MODEL (DUAL OPTIMIZER STEP WITH AMP)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 1, 4, 1, 0)
        )
    def forward(self, x):
        return self.net(x)

class GANAnomaly(BaseAnomalyModel):
    def __init__(self):
        super().__init__()
        self.generator = ConvAutoencoder()
        self.discriminator = Discriminator()
        self.opt_g = None
        self.opt_d = None
        self.scaler_g = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"])
        self.scaler_d = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"])

    def init_optimizers(self, lr):
        if self.opt_g is None or self.opt_d is None:
            self.opt_g = torch.optim.Adam(self.generator.parameters(), lr=lr, betas=(0.5, 0.999))
            self.opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    def forward(self, x: torch.Tensor):
        return self.generator(x)

    def custom_train_step(self, batch, lr=1e-3) -> float:
        self.init_optimizers(lr)
        images = batch if isinstance(batch, torch.Tensor) else batch["image"]

        # 1. Train Discriminator
        self.opt_d.zero_grad()
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"]):
            recon, _ = self.generator(images)
            real_preds = self.discriminator(images)
            fake_preds = self.discriminator(recon.detach())
            loss_d_real = F.binary_cross_entropy_with_logits(real_preds, torch.ones_like(real_preds))
            loss_d_fake = F.binary_cross_entropy_with_logits(fake_preds, torch.zeros_like(fake_preds))
            loss_d = (loss_d_real + loss_d_fake) * 0.5

        self.scaler_d.scale(loss_d).backward()
        self.scaler_d.step(self.opt_d)
        self.scaler_d.update()

        # 2. Train Generator
        self.opt_g.zero_grad()
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"]):
            fake_preds_g = self.discriminator(recon)
            loss_g_adv = F.binary_cross_entropy_with_logits(fake_preds_g, torch.ones_like(fake_preds_g))
            loss_g_recon = F.mse_loss(recon, images)
            loss_g = loss_g_recon + 0.1 * loss_g_adv

        self.scaler_g.scale(loss_g).backward()
        self.scaler_g.step(self.opt_g)
        self.scaler_g.update()

        return loss_g.item()

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            recon, latent = self.generator(x)
            pixel_mse = torch.mean((x - recon) ** 2, dim=1, keepdim=True)
            disc_score = torch.sigmoid(self.discriminator(x))
            disc_map = F.interpolate(1 - disc_score, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

            pixel_err = 0.7 * pixel_mse + 0.3 * disc_map
            image_err = pixel_err.view(pixel_err.size(0), -1).max(dim=1)[0]
            pooled_embed = self.generator.pool(latent).squeeze(-1).squeeze(-1)

            return {
                "image_scores": image_err.cpu().numpy(),
                "pixel_scores": pixel_err.squeeze(1).cpu().numpy(),
                "reconstruction": recon.cpu().numpy(),
                "embeddings": pooled_embed.cpu().numpy()
            }

In [ ]:
# DRAEM (RECONSTRUCTION + SEGMENTATION LOSS)

class DRAEM(BaseAnomalyModel):

  def __init__(self):
    super().__init__()
    self.reconstructive = ConvAutoencoder()
    self.discriminative = nn.Sequential(
        nn.Conv2d(6, 32, 3, 1, 1),
        nn.ReLU(True),
        nn.Conv2d(32, 64, 3, 1, 1),
        nn.ReLU(True),
        nn.Conv2d(64, 32, 3, 1, 1),
        nn.ReLU(True),
        nn.Conv2d(32, 1, 3, 1, 1),  # Sigmoid removed for AMP compatibility
    )

  def _generate_synthetic_anomaly(self, images):
    b, c, h, w = images.shape
    masks = torch.zeros((b, 1, h, w), device=images.device)
    corrupted = images.clone()

    for i in range(b):
      if random.random() > 0.5:
        rx, ry = random.randint(0, w // 2), random.randint(0, h // 2)
        rw, rh = random.randint(w // 8, w // 3), random.randint(h // 8, h // 3)
        masks[i, 0, ry : ry + rh, rx : rx + rw] = 1.0
        noise = torch.rand((c, rh, rw), device=images.device)
        corrupted[i, :, ry : ry + rh, rx : rx + rw] = noise

    return corrupted, masks

  def forward(self, x: torch.Tensor):
    recon, latent = self.reconstructive(x)
    disc_in = torch.cat([x, recon], dim=1)
    anomaly_logits = self.discriminative(disc_in)
    return recon, anomaly_logits, latent

  def compute_loss(self, batch) -> torch.Tensor:
    images = batch if isinstance(batch, torch.Tensor) else batch["image"]
    corrupted_images, target_masks = self._generate_synthetic_anomaly(images)

    recon, latent = self.reconstructive(corrupted_images)
    disc_in = torch.cat([corrupted_images, recon], dim=1)
    pred_logits = self.discriminative(disc_in)

    loss_recon = F.l1_loss(recon, images)
    # BCE with Logits is safe for autocast / mixed-precision training
    loss_seg = F.binary_cross_entropy_with_logits(pred_logits, target_masks)
    return loss_recon + loss_seg

  def predict(self, x: torch.Tensor) -> dict:
    self.eval()
    with torch.no_grad():
      recon, anomaly_logits, latent = self.forward(x)
      pixel_err = torch.sigmoid(anomaly_logits)  # Convert logits to probabilities
      image_err = pixel_err.view(pixel_err.size(0), -1).max(dim=1)[0]
      pooled_embed = self.reconstructive.pool(latent).squeeze(-1).squeeze(-1)

      return {
          "image_scores": image_err.cpu().numpy(),
          "pixel_scores": pixel_err.squeeze(1).cpu().numpy(),
          "reconstruction": recon.cpu().numpy(),
          "embeddings": pooled_embed.cpu().numpy(),
      }

In [ ]:
# CUTPASTE

class CutPasteDetector(BaseAnomalyModel):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.head = nn.Linear(512, 2)

    def _apply_cutpaste(self, x):
        b, c, h, w = x.shape
        x_aug = x.clone()
        for i in range(b):
            rx, ry = random.randint(0, max(1, w//2)), random.randint(0, max(1, h//2))
            rw, rh = random.randint(max(1, w//8), max(2, w//4)), random.randint(max(1, h//8), max(2, h//4))
            patch = x[i, :, ry:ry+rh, rx:rx+rw]
            to_x, to_y = random.randint(0, max(1, w-rw)), random.randint(0, max(1, h-rh))
            x_aug[i, :, to_y:to_y+rh, to_x:to_x+rw] = patch
        return x_aug

    def forward(self, x: torch.Tensor):
        embed = self.backbone(x).squeeze(-1).squeeze(-1)
        logits = self.head(embed)
        return logits, embed

    def compute_loss(self, batch) -> torch.Tensor:
        images = batch if isinstance(batch, torch.Tensor) else batch["image"]
        x_cut = self._apply_cutpaste(images)
        logits_norm, _ = self.forward(images)
        logits_cut, _ = self.forward(x_cut)

        loss = F.cross_entropy(logits_norm, torch.zeros(images.size(0), dtype=torch.long, device=images.device)) + \
               F.cross_entropy(logits_cut, torch.ones(images.size(0), dtype=torch.long, device=images.device))
        return loss

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            logits, embed = self.forward(x)
            probs = F.softmax(logits, dim=1)[:, 1]
            pixel_map = probs.view(-1, 1, 1).repeat(1, x.shape[2], x.shape[3])

            return {
                "image_scores": probs.cpu().numpy(),
                "pixel_scores": pixel_map.cpu().numpy(),
                "reconstruction": None,
                "embeddings": embed.cpu().numpy()
            }

In [ ]:
# PATCHCORE (WITH REGISTERED BUFFER STATE PERSISTENCE)

class PatchCore(BaseAnomalyModel):
    def __init__(self, memory_bank_size=1000):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        self.memory_bank_size = memory_bank_size
        # Register persistent buffer so memory_bank is included in state_dict()
        self.register_buffer("memory_bank", torch.empty(0), persistent=True)

    def fit_memory_bank(self, train_loader, device=DEVICE):
        self.eval()
        self.to(device)
        features_list = []
        with torch.no_grad():
            for batch in train_loader:
                imgs = batch.to(device) if isinstance(batch, torch.Tensor) else batch["image"].to(device)
                feats = self.feature_extractor(imgs)
                feats = feats.permute(0, 2, 3, 1).reshape(-1, feats.shape[1])
                features_list.append(feats.cpu())
                if len(features_list) * feats.shape[0] >= self.memory_bank_size * 5:
                    break

        all_feats = torch.cat(features_list, dim=0)
        idx = torch.randperm(all_feats.shape[0])[:self.memory_bank_size]
        # Re-register or update persistent tensor buffer
        self.register_buffer("memory_bank", all_feats[idx].to(device), persistent=True)

    def forward(self, x: torch.Tensor):
        return self.feature_extractor(x)

    def compute_loss(self, batch) -> torch.Tensor:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            feats = self.feature_extractor(x)
            b, c, h, w = feats.shape

            if self.memory_bank.numel() == 0:
                sampled_bank = feats.permute(0, 2, 3, 1).reshape(-1, c)[:self.memory_bank_size]
                self.register_buffer("memory_bank", sampled_bank, persistent=True)

            feats_flat = feats.permute(0, 2, 3, 1).reshape(-1, c)
            dists = torch.cdist(feats_flat, self.memory_bank, p=2)
            min_dists, _ = torch.min(dists, dim=1)

            pixel_map = min_dists.reshape(b, 1, h, w)
            pixel_map = F.interpolate(pixel_map, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)
            image_scores = pixel_map.view(b, -1).max(dim=1)[0]

            return {
                "image_scores": image_scores.cpu().numpy(),
                "pixel_scores": pixel_map.squeeze(1).cpu().numpy(),
                "reconstruction": None,
                "embeddings": feats.mean(dim=[2,3]).cpu().numpy()
            }

In [ ]:
# FASTFLOW (FEATURE-FLOW BASELINE)

class FastFlowBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels // 2, channels, 3, 1, 1),
            nn.ReLU(True),
            nn.Conv2d(channels, channels, 1, 1, 0)
        )
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        shift_scale = self.conv(x1)
        shift, scale = shift_scale.chunk(2, dim=1)
        scale = torch.tanh(scale)
        y2 = x2 * torch.exp(scale) + shift
        return torch.cat([x1, y2], dim=1), scale

class FastFlow(BaseAnomalyModel):
    """
    Note for Thesis: Simplified, lightweight FastFlow-inspired baseline using a
    pretrained ResNet-18 backbone with single-scale real NVP coupling blocks.
    """
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-4])
        for p in self.feature_extractor.parameters():
            p.requires_grad = False
        self.flow_blocks = nn.ModuleList([FastFlowBlock(128) for _ in range(4)])

    def forward(self, x: torch.Tensor):
        with torch.no_grad():
            feats = self.feature_extractor(x)
        log_jacobians = 0
        z = feats
        for block in self.flow_blocks:
            z, log_j = block(z)
            log_jacobians = log_jacobians + log_j.sum(dim=1, keepdim=True)
        return z, log_jacobians, feats

    def compute_loss(self, batch) -> torch.Tensor:
        images = batch if isinstance(batch, torch.Tensor) else batch["image"]
        z, log_jacobians, _ = self.forward(images)
        nll = 0.5 * (z ** 2) - log_jacobians
        return nll.mean()

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            z, log_jacobians, feats = self.forward(x)
            nll = 0.5 * (z ** 2) - log_jacobians
            pixel_map = F.interpolate(nll.mean(dim=1, keepdim=True), size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)
            image_scores = pixel_map.view(x.size(0), -1).max(dim=1)[0]

            return {
                "image_scores": image_scores.cpu().numpy(),
                "pixel_scores": pixel_map.squeeze(1).cpu().numpy(),
                "reconstruction": None,
                "embeddings": feats.mean(dim=[2,3]).cpu().numpy()
            }

In [ ]:
# CFLOW (CONDITIONAL NORMALIZING FLOW)

class CFLOWCouplingLayer(nn.Module):

  def __init__(self, channels):
    super().__init__()
    self.split_channels = channels // 2
    self.net = nn.Sequential(
        nn.Conv2d(self.split_channels, self.split_channels, 3, 1, 1),
        nn.ReLU(True),
        nn.Conv2d(self.split_channels, self.split_channels * 2, 3, 1, 1),
    )

  def forward(self, x):
    x1, x2 = x.chunk(2, dim=1)
    st = self.net(x1)
    s, t = st.chunk(2, dim=1)
    s = torch.tanh(s)

    y2 = x2 * torch.exp(s) + t
    z = torch.cat([x1, y2], dim=1)

    # Sum log-determinant across split channels to shape (B, 1, H, W)
    log_j = s.sum(dim=1, keepdim=True)
    return z, log_j


class CFLOW(BaseAnomalyModel):

  def __init__(self, in_channels=3, feature_dim=128, num_layers=4):
    super().__init__()
    self.encoder = nn.Sequential(
        nn.Conv2d(in_channels, 32, 4, 2, 1),
        nn.LeakyReLU(0.2, True),
        nn.Conv2d(32, 64, 4, 2, 1),
        nn.LeakyReLU(0.2, True),
        nn.Conv2d(64, feature_dim, 4, 2, 1),
        nn.LeakyReLU(0.2, True),
    )
    self.pool = nn.AdaptiveAvgPool2d((1, 1))

    self.flows = nn.ModuleList(
        [CFLOWCouplingLayer(feature_dim) for _ in range(num_layers)]
    )

  def forward(self, x: torch.Tensor):
    feat = self.encoder(x)
    z = feat
    total_log_j = torch.zeros(
        (x.size(0), 1, feat.size(2), feat.size(3)), device=x.device
    )

    for flow in self.flows:
      z, log_j = flow(z)
      total_log_j = total_log_j + log_j

    return z, total_log_j, feat

  def compute_loss(self, batch) -> torch.Tensor:
    images = batch if isinstance(batch, torch.Tensor) else batch["image"]
    z, log_j, _ = self.forward(images)

    # Guarantee channel dimension broadcasting if log_j is not single-channel
    if log_j.dim() == z.dim() and log_j.size(1) != z.size(1):
      log_j = log_j.sum(dim=1, keepdim=True)

    nll = 0.5 * (z**2) - log_j
    return nll.mean()

  def predict(self, x: torch.Tensor) -> dict:
    self.eval()
    with torch.no_grad():
      z, log_j, feat = self.forward(x)

      # Anomaly metric: high magnitude latent vector + negative log likelihood
      nll = 0.5 * (z**2) - log_j
      pixel_err = nll.mean(dim=1, keepdim=True)

      # Resize heatmap back to input resolution
      pixel_err = F.interpolate(
          pixel_err, size=x.shape[2:], mode="bilinear", align_corners=False
      )
      image_err = pixel_err.view(pixel_err.size(0), -1).max(dim=1)[0]
      pooled_embed = self.pool(feat).squeeze(-1).squeeze(-1)

      return {
          "image_scores": image_err.cpu().numpy(),
          "pixel_scores": pixel_err.squeeze(1).cpu().numpy(),
          "reconstruction": None,
          "embeddings": pooled_embed.cpu().numpy(),
      }

In [ ]:
# HYBRID ANOMALY MODEL

class HybridAnomalyModel(BaseAnomalyModel):
    def __init__(self, alpha=CONFIG["hybrid_alpha"]):
        super().__init__()
        self.autoencoder = ConvAutoencoder()
        self.patchcore = PatchCore()
        self.alpha = alpha

    def forward(self, x: torch.Tensor):
        recon, latent = self.autoencoder(x)
        pc_feats = self.patchcore(x)
        return recon, pc_feats, latent

    def compute_loss(self, batch) -> torch.Tensor:
        return self.autoencoder.compute_loss(batch)

    def fit_memory_bank(self, train_loader, device=DEVICE):
        self.patchcore.fit_memory_bank(train_loader, device=device)

    def predict(self, x: torch.Tensor) -> dict:
        self.eval()
        with torch.no_grad():
            ae_out = self.autoencoder.predict(x)
            pc_out = self.patchcore.predict(x)

            # Independent normalization for every image rather than the whole batch.
            # This is batch-invariant but still requires further score-calibration research.
            def per_image_minmax(v):
                v_min = v.min(axis=(1, 2), keepdims=True)
                v_max = v.max(axis=(1, 2), keepdims=True)
                return (v - v_min) / (v_max - v_min + 1e-8)

            ae_pix = per_image_minmax(ae_out["pixel_scores"])
            pc_pix = per_image_minmax(pc_out["pixel_scores"])

            hybrid_pixel = self.alpha * ae_pix + (1.0 - self.alpha) * pc_pix
            hybrid_image = hybrid_pixel.reshape(x.size(0), -1).max(axis=1)

            return {
                "image_scores": hybrid_image,
                "pixel_scores": hybrid_pixel,
                "reconstruction": ae_out["reconstruction"],
                "embeddings": ae_out["embeddings"]
            }

In [ ]:
# UNIVERSAL TRAINER (WITH AMP SUPPORT)

def train_model(model, train_loader, config=CONFIG, device=DEVICE):
    model.to(device)
    model_name = model.__class__.__name__
    ckpt_dir = os.path.join(config["results_dir"], model_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    best_ckpt_path = os.path.join(ckpt_dir, "best_model.pth")

    print(f"\n--- Processing / Training {model_name} ---")
    start_train_time = time.time()

    if hasattr(model, "fit_memory_bank"):
        print(f"Constructing PatchCore Memory Bank for {model_name}...")
        model.fit_memory_bank(train_loader, device=device)

    # Decoupled check using class name string to prevent NameError
    if model_name == "PatchCore":
        train_duration = time.time() - start_train_time
        torch.save(model.state_dict(), best_ckpt_path)
        return model, [0.0], train_duration

    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=config["lr"]) if not hasattr(model, "custom_train_step") else None
    scaler = torch.cuda.amp.GradScaler(enabled=config["use_amp"]) if optimizer is not None else None
    best_loss = float("inf")
    loss_history = []

    for epoch in range(config["epochs"]):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            if isinstance(batch, torch.Tensor):
                batch = batch.to(device)
            else:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            if hasattr(model, "custom_train_step"):
                loss_val = model.custom_train_step(batch, lr=config["lr"])
            else:
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=config["use_amp"]):
                    loss = model.compute_loss(batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                loss_val = loss.item()

            batch_size = batch.size(0) if isinstance(batch, torch.Tensor) else batch["image"].size(0)
            running_loss += loss_val * batch_size

        epoch_loss = running_loss / len(train_loader.dataset)
        loss_history.append(epoch_loss)
        print(f"Epoch [{epoch+1}/{config['epochs']}] Loss: {epoch_loss:.6f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), best_ckpt_path)

    train_duration = time.time() - start_train_time
    print(f"Loaded best checkpoint from {best_ckpt_path} (Best Loss: {best_loss:.6f} | Total Time: {train_duration:.2f}s)")
    model.load_state_dict(torch.load(best_ckpt_path, map_location=device, weights_only=True))
    return model, loss_history, train_duration

In [ ]:
# METRICS UTILITIES (RAM-SAFE SUBSAMPLING)

def calibrate_thresholds(model, normal_loader, device=DEVICE, image_percentile=99.0,
                         pixel_percentile=99.0, max_images=256, max_pixels_per_image=1024):
  """Estimate decision thresholds on normal-only reference images (no test-label leakage).

  This is a simple research baseline, not a calibrated production decision threshold.
  Image and pixel thresholds are estimated separately, and should be validated
  on independent, held-out normal images in more rigorous experiments.
  """
  model.eval()
  image_scores, pixel_scores = [], []
  count = 0
  rng = np.random.default_rng(CONFIG["seed"])
  with torch.no_grad():
    for batch in normal_loader:
      x = batch.to(device) if isinstance(batch, torch.Tensor) else batch["image"].to(device)
      output = model.predict(x)
      for score, heatmap in zip(output["image_scores"], output["pixel_scores"]):
        image_scores.append(float(score))
        pixels = heatmap.reshape(-1)
        idx = rng.choice(pixels.size, min(max_pixels_per_image, pixels.size), replace=False)
        pixel_scores.extend(pixels[idx].tolist())
        count += 1
        if count >= max_images:
          break
      if count >= max_images:
        break
  if not image_scores or not pixel_scores:
    raise RuntimeError("No normal images available for threshold calibration.")
  return (float(np.percentile(image_scores, image_percentile)),
          float(np.percentile(pixel_scores, pixel_percentile)))


def compute_segmentation_metrics(gt_masks, pixel_scores, threshold):
  bin_preds = (pixel_scores >= threshold).astype(np.uint8)
  bin_gt = (gt_masks > 0.5).astype(np.uint8)

  intersection = np.logical_and(bin_preds, bin_gt).sum()
  union = np.logical_or(bin_preds, bin_gt).sum()

  iou = intersection / union if union > 0 else 0.0
  dice = (
      (2.0 * intersection) / (bin_preds.sum() + bin_gt.sum())
      if (bin_preds.sum() + bin_gt.sum()) > 0
      else 0.0
  )
  return round(float(iou), 4), round(float(dice), 4)


def compute_fast_aupro(
    gt_masks, pixel_scores, max_fpr=0.3, num_thresh=50, max_samples=150
):
  # Filter only defective images and cap samples to prevent RAM spikes
  anom_indices = np.where(gt_masks.max(axis=(1, 2)) > 0)[0]
  if len(anom_indices) == 0:
    return 0.0

  if len(anom_indices) > max_samples:
    np.random.seed(CONFIG["seed"])
    anom_indices = np.random.choice(anom_indices, max_samples, replace=False)

  gt_sub = (gt_masks[anom_indices] > 0.5).astype(np.int8)
  pixel_sub = pixel_scores[anom_indices]

  thresholds = np.linspace(pixel_sub.min(), pixel_sub.max(), num_thresh)
  labeled_masks = [nd_label(m)[0] for m in gt_sub]
  num_regions = sum(np.max(lm) for lm in labeled_masks)

  if num_regions == 0:
    return 0.0

  pros, fprs = [], []
  neg_mask_sum = np.sum(gt_sub == 0)

  for th in thresholds:
    binary_map = (pixel_sub >= th).astype(np.int8)
    fp = np.sum((binary_map == 1) & (gt_sub == 0))
    fprs.append(fp / neg_mask_sum if neg_mask_sum > 0 else 0)

    overlaps = []
    for i, lm in enumerate(labeled_masks):
      n_reg = np.max(lm)
      for r in range(1, n_reg + 1):
        reg_k = lm == r
        overlaps.append(np.sum(binary_map[i][reg_k]) / np.sum(reg_k))
    pros.append(np.mean(overlaps) if overlaps else 0)

  idx = np.argsort(fprs)
  fprs, pros = np.array(fprs)[idx], np.array(pros)[idx]
  valid = fprs <= max_fpr
  return (
      float(np.trapz(pros[valid], fprs[valid]) / max_fpr)
      if np.sum(valid) >= 2
      else 0.0
  )

In [ ]:
# UNIVERSAL EVALUATOR (131M PIXEL ARRAY SUBSAMPLING)

import gc


def evaluate_framework_model(
    model_name,
    model,
    test_loader,
    train_duration=0.0,
    device=DEVICE,
    image_threshold=None,
    pixel_threshold=None,
    warmup_batches=2,
    max_vis_samples=8,
):
  if image_threshold is None or pixel_threshold is None:
    raise ValueError("Pass thresholds from calibrate_thresholds() on normal training images.")
  model.to(device)
  model.eval()

  image_scores_list, pixel_scores_list = [], []
  gt_labels_list, gt_masks_list = [], []
  class_names, latencies = [], []

  sample_images, sample_reconstructions = [], []
  embeddings_list = []
  total_vis_count = 0

  if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(device)

  start_infer_total = time.time()
  with torch.no_grad():
    for i, batch in enumerate(test_loader):
      imgs = batch["image"].to(device)
      labels = batch["label"].numpy()
      masks = batch["mask"].squeeze(1).numpy()

      # Warmup is additional work; do not silently remove test samples.
      if i < warmup_batches:
        _ = model.predict(imgs)
      if torch.cuda.is_available():
        torch.cuda.synchronize()
      start_time = time.perf_counter()
      outputs = model.predict(imgs)
      if torch.cuda.is_available():
        torch.cuda.synchronize()

      latencies.append((time.perf_counter() - start_time) / imgs.size(0))

      image_scores_list.extend(outputs["image_scores"])
      pixel_scores_list.append(outputs["pixel_scores"].astype(np.float32))
      gt_labels_list.extend(labels)
      gt_masks_list.append(masks.astype(np.uint8))
      class_names.extend(batch["class_name"])

      if outputs["embeddings"] is not None:
        embeddings_list.append(outputs["embeddings"])

      if total_vis_count < max_vis_samples:
        sample_images.append(imgs.cpu().numpy())
        if outputs["reconstruction"] is not None:
          sample_reconstructions.append(outputs["reconstruction"])
        total_vis_count += imgs.size(0)

  total_infer_duration = time.time() - start_infer_total
  eval_df = pd.DataFrame({
      "class_name": class_names,
      "gt_label": gt_labels_list,
      "image_score": image_scores_list,
  })

  class_metrics = []
  for c_name, group in eval_df.groupby("class_name"):
    if len(group["gt_label"].unique()) > 1:
      c_auroc = roc_auc_score(group["gt_label"], group["image_score"])
      class_metrics.append({
          "Model": model_name,
          "Category": c_name,
          "Image AUROC": round(float(c_auroc), 4),
      })

  per_class_df = pd.DataFrame(class_metrics)

  gt_labels = np.array(gt_labels_list)
  gt_masks = np.concatenate(gt_masks_list, axis=0)
  image_scores = np.array(image_scores_list)
  pixel_scores = np.concatenate(pixel_scores_list, axis=0)

  opt_thresh = float(image_threshold)
  preds = (image_scores >= opt_thresh).astype(int)

  img_auroc = roc_auc_score(gt_labels, image_scores)

  # Subsample 131M pixels by a factor of 8 to evaluate Pixel AUROC without OOM crash
  pixel_gt_flat = gt_masks.reshape(-1)[::8]
  pixel_scores_flat = pixel_scores.reshape(-1)[::8]
  pixel_auroc = roc_auc_score(pixel_gt_flat, pixel_scores_flat)

  aupro = compute_fast_aupro(gt_masks, pixel_scores)
  precision = precision_score(gt_labels, preds, zero_division=0)
  recall = recall_score(gt_labels, preds, zero_division=0)
  f1 = f1_score(gt_labels, preds, zero_division=0)
  iou, dice = compute_segmentation_metrics(gt_masks, pixel_scores, float(pixel_threshold))

  avg_latency_ms = np.mean(latencies) * 1000
  fps = 1000.0 / avg_latency_ms if avg_latency_ms > 0 else 0.0

  alloc_mem = (
      torch.cuda.max_memory_allocated(device) / (1024**2)
      if torch.cuda.is_available()
      else 0.0
  )
  res_mem = (
      torch.cuda.max_memory_reserved(device) / (1024**2)
      if torch.cuda.is_available()
      else 0.0
  )
  total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  model_size_mb = model.get_model_size_mb()

  overall_df = pd.DataFrame([{
      "Model": model_name,
      "Category": "Mean / Overall",
      "Image AUROC": round(float(img_auroc), 4),
      "Pixel AUROC": round(float(pixel_auroc), 4),
      "AUPRO": round(float(aupro), 4),
      "Precision": round(float(precision), 4),
      "Recall": round(float(recall), 4),
      "F1 Score": round(float(f1), 4),
      "IoU": round(float(iou), 4),
      "Dice": round(float(dice), 4),
      "Train Time (s)": round(float(train_duration), 2),
      "Total Infer Time (s)": round(float(total_infer_duration), 2),
      "Latency (ms)": round(float(avg_latency_ms), 2),
      "FPS": round(float(fps), 1),
      "Model Size (MB)": round(float(model_size_mb), 2),
      "Mem Allocated (MB)": round(float(alloc_mem), 2),
      "Mem Reserved (MB)": round(float(res_mem), 2),
      "Params": total_params,
  }])

  eval_results = {
      "gt_labels": gt_labels,
      "gt_masks": gt_masks,
      "image_scores": image_scores,
      "pixel_scores": pixel_scores,
      "images": (
          np.concatenate(sample_images, axis=0)[:max_vis_samples]
          if sample_images
          else None
      ),
      "reconstructions": (
          np.concatenate(sample_reconstructions, axis=0)[:max_vis_samples]
          if sample_reconstructions
          else None
      ),
      "embeddings": (
          np.concatenate(embeddings_list, axis=0) if embeddings_list else None
      ),
      "opt_thresh": opt_thresh,
      "pixel_threshold": float(pixel_threshold),
      "per_class_df": per_class_df,
  }

  combined_df = pd.concat([overall_df, per_class_df], ignore_index=True)
  return combined_df, eval_results

In [ ]:
#COMPLETE VISUALIZATION & EXPORTER UTILITIES

def plot_tsne_embeddings(
    embeddings, gt_labels, model_name, max_samples=CONFIG["max_tsne_samples"]
):
  if embeddings is None or len(embeddings) < 10:
    return

  labels = gt_labels[: len(embeddings)]
  if len(embeddings) > max_samples:
    idx = np.random.choice(len(embeddings), max_samples, replace=False)
    embeddings = embeddings[idx]
    labels = labels[idx]

  perp = min(30, max(1, len(embeddings) - 1))
  tsne = TSNE(n_components=2, perplexity=perp, random_state=CONFIG["seed"])
  tsne_features = tsne.fit_transform(embeddings)

  fig = plt.figure(figsize=(7, 5))
  sns.scatterplot(
      x=tsne_features[:, 0],
      y=tsne_features[:, 1],
      hue=labels,
      palette={0: "blue", 1: "red"},
      alpha=0.7,
  )
  plt.title(f"{model_name} Feature Embedding (t-SNE)")
  plt.xlabel("Component 1")
  plt.ylabel("Component 2")
  plt.grid(True)
  plt.show()
  plt.close(fig)


def visualize_results(model_name, metrics_df, eval_results, loss_history=None):
  print("=" * 80)
  print(model_name)
  print("=" * 80)

  display(metrics_df)

  gt_labels = eval_results["gt_labels"]
  gt_masks = eval_results["gt_masks"]
  image_scores = eval_results["image_scores"]
  pixel_scores = eval_results["pixel_scores"]
  images = eval_results["images"]
  reconstructions = eval_results["reconstructions"]
  embeddings = eval_results["embeddings"]
  opt_thresh = eval_results["opt_thresh"]

  if loss_history is not None:
    fig_loss = plt.figure(figsize=(7, 4))
    plt.plot(loss_history, linewidth=2)
    plt.title(f"{model_name} Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()
    plt.close(fig_loss)

  fig_grid, axes = plt.subplots(2, 2, figsize=(12, 9))

  fpr_img, tpr_img, _ = roc_curve(gt_labels, image_scores)

  # Subsample flattened pixels for ROC plot
  fpr_pix, tpr_pix, _ = roc_curve(
      gt_masks.reshape(-1)[::8], pixel_scores.reshape(-1)[::8]
  )

  axes[0, 0].plot(fpr_img, tpr_img, linewidth=2, label="Image ROC")
  axes[0, 0].plot(fpr_pix, tpr_pix, linewidth=2, label="Pixel ROC")
  axes[0, 0].plot([0, 1], [0, 1], "k--")
  axes[0, 0].set_title("ROC Curves")
  axes[0, 0].legend()
  axes[0, 0].grid(True)

  precision, recall, _ = precision_recall_curve(gt_labels, image_scores)
  axes[0, 1].plot(recall, precision, linewidth=2, color="purple")
  axes[0, 1].set_title("Precision Recall Curve")
  axes[0, 1].grid(True)

  predictions = (image_scores >= opt_thresh).astype(int)
  cm = confusion_matrix(gt_labels, predictions)
  ConfusionMatrixDisplay(
      confusion_matrix=cm, display_labels=["Normal", "Defect"]
  ).plot(ax=axes[1, 0], cmap="Blues", colorbar=False)
  axes[1, 0].set_title("Confusion Matrix")

  sns.histplot(
      x=image_scores,
      hue=gt_labels,
      kde=True,
      palette={0: "green", 1: "red"},
      ax=axes[1, 1],
  )
  axes[1, 1].axvline(opt_thresh, color="black", linestyle="--")
  axes[1, 1].set_title("Anomaly Score Distribution")

  plt.tight_layout()
  plt.show()
  plt.close(fig_grid)

  plot_tsne_embeddings(embeddings, gt_labels, model_name)

  if images is not None:
    n = min(4, len(images))
    cols = 6 if reconstructions is not None else 4
    fig_vis, axes = plt.subplots(n, cols, figsize=(3 * cols, 3 * n))
    if n == 1:
      axes = np.expand_dims(axes, 0)

    mean = np.array(CONFIG["imagenet_mean"])
    std = np.array(CONFIG["imagenet_std"])

    for i in range(n):
      original = np.clip(images[i].transpose(1, 2, 0) * std + mean, 0, 1)
      gt = gt_masks[i]
      heatmap = pixel_scores[i]

      axes[i, 0].imshow(original)
      axes[i, 0].set_title("Original")
      axes[i, 1].imshow(gt, cmap="gray")
      axes[i, 1].set_title("Ground Truth")
      axes[i, 2].imshow(heatmap, cmap="jet")
      axes[i, 2].set_title("Heatmap")
      axes[i, 3].imshow(original)
      axes[i, 3].imshow(heatmap, cmap="jet", alpha=0.5)
      axes[i, 3].set_title("Overlay")

      if reconstructions is not None:
        reconstruction = np.clip(
            reconstructions[i].transpose(1, 2, 0) * std + mean, 0, 1
        )
        difference = np.abs(original - reconstruction).mean(axis=-1)
        axes[i, 4].imshow(reconstruction)
        axes[i, 4].set_title("Reconstruction")
        axes[i, 5].imshow(difference, cmap="magma")
        axes[i, 5].set_title("Pixel Difference")

      for ax in axes[i]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig_vis)

  plt.close("all")


def save_and_visualize_results(
    model_name,
    metrics_df,
    eval_results,
    loss_history=None,
    save_dir=CONFIG["results_dir"],
):
  out_dir = os.path.join(save_dir, model_name)
  os.makedirs(out_dir, exist_ok=True)

  metrics_df.to_csv(os.path.join(out_dir, "Metrics.csv"), index=False)
  visualize_results(
      model_name, metrics_df, eval_results, loss_history=loss_history
  )

In [ ]:
# PIPELINE RUNNER & COMPARISON EXPORTER

master_results_df = pd.DataFrame()


def append_to_comparison_table(metrics_df):
  global master_results_df
  master_results_df = pd.concat(
      [master_results_df, metrics_df], ignore_index=True
  )
  master_results_df.to_csv(
      os.path.join(CONFIG["results_dir"], "Master_Model_Comparison.csv"),
      index=False,
  )
  return master_results_df


def save_model_checkpoint(model, model_name):
  path = os.path.join(
      CONFIG["results_dir"], model_name, f"{model_name}_final.pth"
  )
  torch.save(model.state_dict(), path)

In [ ]:
trained_model, loss_hist, train_dur = train_model(ConvAutoencoder(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("Autoencoder", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("Autoencoder", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "Autoencoder")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(GANAnomaly(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("GAN", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("GAN", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "GAN")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(CutPasteDetector(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("CutPaste", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("CutPaste", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "CutPaste")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(PatchCore(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("PatchCore", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("PatchCore", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "PatchCore")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(FastFlow(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("FastFlow", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("FastFlow", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "FastFlow")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

Intermediate results appear in `master_results_df` after the corresponding cells have run. Previous stale outputs were removed to avoid presenting mixed execution histories.


Intermediate results appear in `master_results_df` after the corresponding cells have run. Previous stale outputs were removed to avoid presenting mixed execution histories.


In [ ]:
trained_model, loss_hist, train_dur = train_model(DRAEM(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("DRAEM", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("DRAEM", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "DRAEM")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(CFLOW(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("CFLOW", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("CFLOW", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "CFLOW")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

In [ ]:
trained_model, loss_hist, train_dur = train_model(HybridAnomalyModel(), train_loader, config=CONFIG, device=DEVICE)

In [ ]:
image_threshold, pixel_threshold = calibrate_thresholds(trained_model, calibration_loader, device=DEVICE)
metrics_df, eval_data = evaluate_framework_model("Hybrid", trained_model, test_loader, train_duration=train_dur, device=DEVICE, image_threshold=image_threshold, pixel_threshold=pixel_threshold)

save_and_visualize_results("Hybrid", metrics_df, eval_data, loss_history=loss_hist)
save_model_checkpoint(trained_model, "Hybrid")
summary_table = append_to_comparison_table(metrics_df[metrics_df["Category"] == "Mean / Overall"])

Intermediate results appear in `master_results_df` after the corresponding cells have run. Previous stale outputs were removed to avoid presenting mixed execution histories.


In [ ]:
# Use measured data from this run rather than manually typed, inconsistent figures.
if master_results_df.empty:
    raise RuntimeError("Train and evaluate at least one model first.")
df = master_results_df.drop_duplicates(subset=["Model"], keep="last").copy()
df = df.sort_values(by=["Pixel AUROC", "AUPRO"], ascending=[False, False]).reset_index(drop=True)
display(df)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
palette = sns.color_palette("mako", n_colors=len(df))

# 1. Pixel AUROC
sns.barplot(
    data=df,
    x="Pixel AUROC",
    y="Model",
    ax=axes[0, 0],
    palette="viridis",
    hue="Model",
    legend=False,
)
axes[0, 0].set_title("Pixel AUROC (Higher is Better)", fontweight="bold")
axes[0, 0].set_xlim(0.4, 0.85)

# 2. AUPRO
sns.barplot(
    data=df,
    x="AUPRO",
    y="Model",
    ax=axes[0, 1],
    palette="rocket",
    hue="Model",
    legend=False,
)
axes[0, 1].set_title("AUPRO (Area Under PRO)", fontweight="bold")
axes[0, 1].set_xlim(0.0, 0.6)

# 3. F1 Score
sns.barplot(
    data=df,
    x="F1 Score",
    y="Model",
    ax=axes[1, 0],
    palette="crest",
    hue="Model",
    legend=False,
)
axes[1, 0].set_title("Classification F1 Score", fontweight="bold")
axes[1, 0].set_xlim(0.0, 1.0)

# 4. IoU
sns.barplot(
    data=df,
    x="IoU",
    y="Model",
    ax=axes[1, 1],
    palette="magma",
    hue="Model",
    legend=False,
)
axes[1, 1].set_title("Segmentation IoU", fontweight="bold")
axes[1, 1].set_xlim(0.0, 0.08)

for ax in axes.flat:
  ax.set_ylabel("")
  for p in ax.patches:
    width = p.get_width()
    if width > 0:
      ax.annotate(
          f"{width:.4f}",
          (width, p.get_y() + p.get_height() / 2.0),
          ha="left",
          va="center",
          xytext=(5, 0),
          textcoords="offset points",
          fontsize=9,
      )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 7))

# Bubble plot: X=FPS, Y=Pixel AUROC, Size=Model Size
scatter = sns.scatterplot(
    data=df,
    x="FPS",
    y="Pixel AUROC",
    size="Model Size (MB)",
    hue="Model",
    sizes=(100, 1000),
    palette="tab10",
    alpha=0.85,
)

# Annotate each point with model name
for i in range(len(df)):
  plt.text(
      df["FPS"][i] + 15,
      df["Pixel AUROC"][i] + 0.003,
      df["Model"][i],
      fontweight="bold",
      fontsize=10,
  )

plt.title(
    "Localization Quality vs. Throughput (Bubble Size = Model Size MB)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Inference Speed (FPS) →", fontsize=11, fontweight="bold")
plt.ylabel("Pixel AUROC →", fontsize=11, fontweight="bold")
plt.xlim(left=0)
plt.ylim(0.50, 0.82)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4.5))
df_train_sort = df.sort_values(by="Train Time (s)")
ax = sns.barplot(
    data=df_train_sort,
    x="Train Time (s)",
    y="Model",
    palette="Spectral",
    hue="Model",
    legend=False,
)

plt.xscale("log")  # Log scale to visually emphasize PatchCore speedup
plt.title(
    "Training Duration Comparison (Log Scale)", fontsize=12, fontweight="bold"
)
plt.xlabel("Training Time (seconds - Log Scale)", fontsize=10)
plt.ylabel("")

for p in ax.patches:
  val = p.get_width()
  ax.annotate(
      f"{val:.2f}s",
      (val, p.get_y() + p.get_height() / 2.0),
      ha="left",
      va="center",
      xytext=(6, 0),
      textcoords="offset points",
      fontsize=9,
      fontweight="bold",
  )

plt.tight_layout()
plt.show()